# Cleaning_CYRV_orders_dataset.csv

## Conclusion

No rows were removed and no missing or inconsistent timestamps were imputed because there was insufficient evidence to determine their correct values.

The cleaned dataset preserves the original order records while converting date fields to appropriate datetime format during the cleaning process and documenting known temporal inconsistencies for downstream analysis.

The cleaned dataset contains 99,441 rows and 8 columns and is ready for downstream analysis, with the documented timestamp limitations taken into account.

## Cleaning Summary

The `CYRV_orders_dataset.csv` dataset was inspected and validated for structural consistency, missing values, order statuses, timestamp integrity, and customer relationships.

### Dataset Structure
- Original shape: 99,441 rows × 8 columns
- Duplicate rows: 0
- Duplicate `order_id` values: 0
- Duplicate `customer_id` values: 0
- All 99,441 orders have unique order IDs and customer IDs.

### Missing Values
Missing values were identified only in lifecycle timestamp columns:
- `order_approved_at`: 160
- `order_delivered_carrier_date`: 1,783
- `order_delivered_customer_date`: 2,965

Most missing timestamps are consistent with the corresponding order status and therefore represent incomplete stages of the order lifecycle rather than necessarily invalid data.

Among delivered orders, 23 records contain at least one missing approval, carrier, or customer delivery timestamp. These records were retained because the correct missing timestamps cannot be reliably reconstructed.

### Date Validation
All five date columns were successfully converted to datetime format without introducing additional missing values.

Chronological validation identified several timestamp inconsistencies:
- 166 orders have a carrier delivery timestamp earlier than the purchase timestamp.
- 23 orders have a customer delivery timestamp earlier than the carrier delivery timestamp.
- 1,359 orders have a carrier delivery timestamp earlier than the approval timestamp.
- 61 orders have a customer delivery timestamp earlier than the approval timestamp.

Most carrier-before-purchase cases involve relatively small time differences, although two substantial outliers were identified. Since the correct timestamps cannot be determined from the available data, these records were retained rather than manually corrected or removed.

### Estimated Delivery Date
All 99,441 `order_estimated_delivery_date` values are recorded at `00:00:00`.

Therefore, this field should be interpreted primarily as an estimated calendar delivery date rather than an exact delivery timestamp. Analyses of on-time delivery should compare calendar dates rather than exact timestamps.

### Customer Relationship Validation
The orders dataset was cross-checked against the cleaned customers dataset:
- Unique customer IDs in orders: 99,441
- Unique customer IDs in customers: 99,441
- Order customer IDs missing from customers: 0
- Customer IDs without a corresponding order: 0

This confirms complete customer ID consistency between the two datasets.


In [27]:
import pandas as pd

orders = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_orders_dataset.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [28]:
print("Shape:", orders.shape)

print("\nData types:")
print(orders.dtypes)

Shape: (99441, 8)

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [29]:
print("Missing values:")
print(orders.isna().sum())

print("\nDuplicate rows:")
print(orders.duplicated().sum())

print("\nUnique order IDs:")
print(orders["order_id"].nunique())

print("\nDuplicate order IDs:")
print(orders["order_id"].duplicated().sum())

print("\nUnique customer IDs:")
print(orders["customer_id"].nunique())

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate rows:
0

Unique order IDs:
99441

Duplicate order IDs:
0

Unique customer IDs:
99441


In [30]:
print(orders["order_status"].value_counts(dropna=False))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [31]:
missing_dates_by_status = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        missing_approved_at=("order_approved_at", lambda x: x.isna().sum()),
        missing_carrier_date=("order_delivered_carrier_date", lambda x: x.isna().sum()),
        missing_customer_date=("order_delivered_customer_date", lambda x: x.isna().sum())
    )
)

missing_dates_by_status

,total_orders,missing_approved_at,missing_carrier_date,missing_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [32]:
delivered_missing_dates = orders[
    (orders["order_status"] == "delivered")
    & (
        orders["order_approved_at"].isna()
        | orders["order_delivered_carrier_date"].isna()
        | orders["order_delivered_customer_date"].isna()
    )
]

print("Delivered orders with missing dates:", len(delivered_missing_dates))

delivered_missing_dates[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

Delivered orders with missing dates: 23


,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


In [33]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print(orders[date_columns].dtypes)

print("\nMissing values after datetime conversion:")
print(orders[date_columns].isna().sum())

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Missing values after datetime conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [34]:
chronology_checks = {
    "approved_before_purchase": (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    ).sum(),

    "carrier_before_purchase": (
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"]
    ).sum(),

    "customer_before_purchase": (
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"]
    ).sum(),

    "customer_before_carrier": (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum()
}

pd.Series(chronology_checks)

approved_before_purchase      0
carrier_before_purchase     166
customer_before_purchase      0
customer_before_carrier      23
dtype: int64

In [35]:
carrier_before_purchase = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
].copy()

carrier_before_purchase["difference_hours"] = (
    carrier_before_purchase["order_purchase_timestamp"]
    - carrier_before_purchase["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

print(carrier_before_purchase["difference_hours"].describe())

carrier_before_purchase[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "difference_hours"
    ]
].sort_values("difference_hours", ascending=False).head(10)

count     166.000000
mean       26.039478
std       318.923845
min         0.006667
25%         0.234444
50%         0.599861
75%         1.017500
max      4109.098056
Name: difference_hours, dtype: float64


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,difference_hours
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-01-26 13:35:00,4109.098056
83321,4021cd7611d6d9ce5ffcd24817fc374f,delivered,2018-08-18 11:49:40,2018-08-14 06:22:00,101.461111
67844,db090a16182b263b1e896bb26c6f66cf,delivered,2018-07-13 16:14:08,2018-07-13 13:59:00,2.252222
65452,9711d975b961355b4b5d636857e48498,delivered,2018-06-13 15:23:50,2018-06-13 13:15:00,2.147222
79401,89d32b64af005178b318f76cd60f2c3c,delivered,2018-07-06 11:54:40,2018-07-06 09:48:00,2.111111
13969,6192897f85cb2aff6dd1fca56fddb45e,delivered,2018-07-18 13:34:22,2018-07-18 11:33:00,2.022778
43657,3ed559dec69ab96d9074fedaf18650eb,delivered,2018-06-15 13:20:09,2018-06-15 11:22:00,1.969167
4256,4e157a36ea9cf89bde6fff57a780b525,delivered,2018-08-24 14:37:50,2018-08-24 12:43:00,1.913889
74503,119060af27b04f5fae8b2b0e27eef7f9,delivered,2018-06-11 14:34:47,2018-06-11 12:45:00,1.829722
21910,e23c9dcc304042fa90d538be679fa68d,delivered,2018-06-13 11:41:23,2018-06-13 09:53:00,1.806389


In [36]:
carrier_before_purchase[
    carrier_before_purchase["difference_hours"] > 24
][
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "difference_hours"
    ]
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,difference_hours
25883,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,4109.098056
83321,4021cd7611d6d9ce5ffcd24817fc374f,ac27243bffbc9c58eb62adfb58c71488,delivered,2018-08-18 11:49:40,2018-08-18 12:10:30,2018-08-14 06:22:00,2018-08-29 04:41:53,2018-08-30,101.461111


In [37]:
customer_before_carrier = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
].copy()

customer_before_carrier["difference_hours"] = (
    customer_before_carrier["order_delivered_carrier_date"]
    - customer_before_carrier["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print(customer_before_carrier["difference_hours"].describe())

customer_before_carrier[
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "difference_hours"
    ]
].sort_values("difference_hours", ascending=False).head(10)

count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: difference_hours, dtype: float64


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,difference_hours
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-03-02 17:34:26,2017-02-14 15:15:57,386.308056
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01,189.095000
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-06-09 15:07:29,2017-06-02 11:09:16,171.970278
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,144.238889
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-26 11:43:06,2016-10-20 18:03:17,137.663611
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-26 11:42:05,2016-10-20 19:07:54,136.569722
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-15 08:55:26,2017-02-10 07:58:32,120.948333
41636,b866af202be0692766081310cd4085e1,delivered,2017-02-20 02:32:08,2017-02-15 03:53:46,118.639444
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56,69.417222
78556,ea1dcb4757a844d2642547797bd5feb0,delivered,2017-07-27 19:21:31,2017-07-25 19:43:10,47.639167


In [38]:
approval_carrier_checks = {
    "carrier_before_approval": (
        orders["order_delivered_carrier_date"]
        < orders["order_approved_at"]
    ).sum(),

    "customer_before_approval": (
        orders["order_delivered_customer_date"]
        < orders["order_approved_at"]
    ).sum()
}

pd.Series(approval_carrier_checks)

carrier_before_approval     1359
customer_before_approval      61
dtype: int64

In [39]:
carrier_before_approval = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_approved_at"]
].copy()

carrier_before_approval["difference_hours"] = (
    carrier_before_approval["order_approved_at"]
    - carrier_before_approval["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

customer_before_approval = orders[
    orders["order_delivered_customer_date"]
    < orders["order_approved_at"]
].copy()

customer_before_approval["difference_hours"] = (
    customer_before_approval["order_approved_at"]
    - customer_before_approval["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print("Carrier before approval:")
print(carrier_before_approval["difference_hours"].describe())

print("\nCustomer before approval:")
print(customer_before_approval["difference_hours"].describe())


Carrier before approval:
count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
max      4109.256111
Name: difference_hours, dtype: float64

Customer before approval:
count     61.000000
mean      39.054408
std       42.976017
min        0.088611
25%        3.452222
50%       24.823333
75%       55.826944
max      167.751667
Name: difference_hours, dtype: float64


In [40]:
estimated_times = orders["order_estimated_delivery_date"].dt.time

print("Unique estimated delivery times:", estimated_times.nunique())
print("\nMost common estimated delivery times:")
print(estimated_times.value_counts().head())

Unique estimated delivery times: 1

Most common estimated delivery times:
order_estimated_delivery_date
00:00:00    99441
Name: count, dtype: int64


In [41]:
for col in date_columns:
    print(f"{col}:")
    print("Min:", orders[col].min())
    print("Max:", orders[col].max())
    print()

order_purchase_timestamp:
Min: 2016-09-04 21:15:19
Max: 2018-10-17 17:30:18

order_approved_at:
Min: 2016-09-15 12:16:38
Max: 2018-09-03 17:40:06

order_delivered_carrier_date:
Min: 2016-10-08 10:34:01
Max: 2018-09-11 19:48:28

order_delivered_customer_date:
Min: 2016-10-11 13:46:32
Max: 2018-10-17 13:22:46

order_estimated_delivery_date:
Min: 2016-09-30 00:00:00
Max: 2018-11-12 00:00:00



In [42]:
customers = pd.read_csv("../CYRV_customers_cleaned.csv")

orders_customer_ids = set(orders["customer_id"])
customers_customer_ids = set(customers["customer_id"])

print("Customer IDs in orders:", len(orders_customer_ids))
print("Customer IDs in customers:", len(customers_customer_ids))

print(
    "Order customer IDs missing from customers:",
    len(orders_customer_ids - customers_customer_ids)
)

print(
    "Customer IDs without an order:",
    len(customers_customer_ids - orders_customer_ids)
)


Customer IDs in orders: 99441
Customer IDs in customers: 99441
Order customer IDs missing from customers: 0
Customer IDs without an order: 0


In [43]:
print("Shape:", orders.shape)
print("Duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())
print("Duplicate customer IDs:", orders["customer_id"].duplicated().sum())

print("\nMissing values:")
print(orders.isna().sum())

print("\nOrder status counts:")
print(orders["order_status"].value_counts())

Shape: (99441, 8)
Duplicate rows: 0
Duplicate order IDs: 0
Duplicate customer IDs: 0

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [44]:
output_path = "../CYRV_orders_cleaned.csv"

orders.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")


Saved cleaned dataset to: ../CYRV_orders_cleaned.csv


In [45]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_check = pd.read_csv(
    "../CYRV_orders_cleaned.csv",
    parse_dates=date_columns
)

print("Shape:", orders_check.shape)
print("Duplicate rows:", orders_check.duplicated().sum())
print("Duplicate order IDs:", orders_check["order_id"].duplicated().sum())
print("Duplicate customer IDs:", orders_check["customer_id"].duplicated().sum())

print("\nMissing values:")
print(orders_check.isna().sum())

print("\nData types:")
print(orders_check.dtypes)

Shape: (99441, 8)
Duplicate rows: 0
Duplicate order IDs: 0
Duplicate customer IDs: 0

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Data types:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
